# 0. 环境与路径检查

本节确认当前 Notebook 是否运行在正确的 Python/Jupyter 环境，并固定本轮协议要求的新项目路径。输入是当前 Python/Jupyter 环境；输出是真实工程根目录、Notebook 代码目录、testdata 与 src 路径。若这里失败，请先检查 conda kernel、项目目录和依赖安装。

In [ ]:
# 这一步检查解释器、项目路径和关键目录。
# 输入：当前 Jupyter kernel。
# 输出：PROJECT_ROOT、PYTHON_DIR、TESTDATA_DIR 等绝对路径。
# 若失败：确认 Notebook kernel 是否选择 PPG_sensor_env，且 D 盘项目路径存在。
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"D:\HeartDecode\outline-PPGtoHR-main")
PYTHON_DIR = PROJECT_ROOT / "python_notebook_base"
TESTDATA_DIR = PYTHON_DIR / "testdata"
SRC_DIR = PYTHON_DIR / "src"

assert PROJECT_ROOT.exists(), f"工程根目录不存在: {PROJECT_ROOT}"
assert PYTHON_DIR.exists(), f"Notebook 基础目录不存在: {PYTHON_DIR}"
assert TESTDATA_DIR.exists(), f"测试数据目录不存在: {TESTDATA_DIR}"
assert SRC_DIR.exists(), f"源码目录不存在: {SRC_DIR}"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PYTHON_DIR:", PYTHON_DIR)
print("TESTDATA_DIR:", TESTDATA_DIR)
# 本 cell 结束后，后续所有路径都只使用上述新 D 盘路径。

# 1. 导入库与项目模块

本节导入协议运行所需的第三方库和 `ppg_hr.experimental` 模块。输入是已加入 `sys.path` 的源码目录；输出是可调用的批处理、配对、QC、分段与对齐函数。若导入失败，请检查是否在 `PPG_sensor_env` 中安装了项目依赖。

In [ ]:
# 这一步导入依赖和项目函数。
# 输入：PPG_sensor_env 中已安装的 numpy/pandas/scipy 等依赖；推荐安装 optuna。
# 输出：后续 cell 可直接调用的函数和类。
# 若失败：确认缺失库后再安装，例如 optuna、scikit-learn、scipy、matplotlib。
import json
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import Image, display

try:
    import optuna
    print("optuna", optuna.__version__)
except ModuleNotFoundError:
    optuna = None
    print("未安装 optuna：核心模块会使用确定性随机搜索 fallback；正式贝叶斯优化请先安装 optuna。")

from ppg_hr.experimental.alignment import align_ppg_to_ref_hr
from ppg_hr.experimental.batch_pairing import discover_sample_pairs_with_unpaired
from ppg_hr.experimental.preprocess_protocol import load_and_preprocess_protocol, resample_protocol_dataset
from ppg_hr.experimental.qc import quality_filter_sample
from ppg_hr.experimental.run_batch_protocol import run_batch_adaptive_protocol
from ppg_hr.experimental.segmentation import detect_activity_segments
from ppg_hr.params import ProtocolSearchParams

# 本 cell 结束后，协议入口 run_batch_adaptive_protocol 已准备好。

# 2. 配置输入输出路径

本节声明 testdata 输入目录和协议输出目录。输入是 `PYTHON_DIR/testdata`；输出是 CSV、JSON/report、13 路滤波图、HR 对比图、贝叶斯训练曲线图目录。若路径异常，请检查是否仍误用了旧 C 盘路径。

In [ ]:
# 这一步配置输出目录。
# 输入：PYTHON_DIR 和 TESTDATA_DIR。
# 输出：OUTPUT_ROOT、CSV_OUT_DIR、REPORT_OUT_DIR、FILTERED_SIGNAL_OUT_DIR、HR_COMPARE_OUT_DIR、BAYES_CURVE_OUT_DIR。
# 若失败：确认 python_notebook_base/outputs 目录可写。
OUTPUT_ROOT = PYTHON_DIR / "outputs" / "batch_adaptive_protocol"
CSV_OUT_DIR = OUTPUT_ROOT / "csv"
REPORT_OUT_DIR = OUTPUT_ROOT / "report"
FIG_OUT_DIR = OUTPUT_ROOT
FILTERED_SIGNAL_OUT_DIR = OUTPUT_ROOT / "filtered_motion_signals"
HR_COMPARE_OUT_DIR = OUTPUT_ROOT / "hr_compare"
BAYES_CURVE_OUT_DIR = OUTPUT_ROOT / "bayes_training_curves"

for d in [CSV_OUT_DIR, REPORT_OUT_DIR, FILTERED_SIGNAL_OUT_DIR, HR_COMPARE_OUT_DIR, BAYES_CURVE_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("输入目录:", TESTDATA_DIR)
print("CSV 输出:", CSV_OUT_DIR)
print("JSON/report 输出:", REPORT_OUT_DIR)
print("13 路滤波图输出:", FILTERED_SIGNAL_OUT_DIR)
print("HR 对比图输出:", HR_COMPARE_OUT_DIR)
print("贝叶斯训练曲线输出:", BAYES_CURVE_OUT_DIR)
# 本 cell 结束后，所有输出都会落在 python_notebook_base/outputs/batch_adaptive_protocol 下。

# 3. 可选清空旧输出文件

本节只在 `CLEAN_OUTPUTS = True` 时删除协议输出目录中的旧 CSV、JSON、PNG，不触碰 `testdata`、源码或 Notebook。输入是上一步的输出目录；输出是可选的干净输出目录。若安全断言失败，请检查目录是否被误设成源码或测试数据目录。

In [ ]:
# 这一步安全清理旧输出。
# 输入：CLEAN_OUTPUTS、CSV_OUT_DIR、REPORT_OUT_DIR 和三个图片目录。
# 输出：按需删除旧 csv/json/png 后的输出目录。
# 若失败：检查路径断言，避免误删原始 testdata 或源码。
CLEAN_OUTPUTS = True  # 如需重新生成全部结果，手动改为 True；绝不会清理 testdata。

def clear_previous_outputs(*dirs):
    protected = {TESTDATA_DIR.resolve(), PYTHON_DIR.resolve(), PROJECT_ROOT.resolve(), SRC_DIR.resolve()}
    for d in dirs:
        d = Path(d).resolve()
        assert d not in protected, f"拒绝清理受保护目录: {d}"
        assert OUTPUT_ROOT.resolve() in [d, *d.parents], f"拒绝清理协议输出目录之外的路径: {d}"
        assert "testdata" not in {part.lower() for part in d.parts}, f"拒绝清理包含 testdata 的路径: {d}"
        if not d.exists():
            continue
        for pattern in ["*.csv", "*.json", "*.png"]:
            for p in d.glob(pattern):
                p.unlink()

if CLEAN_OUTPUTS:
    clear_previous_outputs(CSV_OUT_DIR, REPORT_OUT_DIR, FILTERED_SIGNAL_OUT_DIR, HR_COMPARE_OUT_DIR, BAYES_CURVE_OUT_DIR)
    print("已清理旧输出:", datetime.now().isoformat(timespec="seconds"))
else:
    print("CLEAN_OUTPUTS=False，本次保留旧输出文件。")
# 本 cell 结束后，只会在 CLEAN_OUTPUTS=True 时删除协议输出目录中的 csv/json/png。

# 4. 扫描并配对测试数据

本节扫描 `testdata` 中的 `multi_*.csv` 与 `_ref.csv`。输入是测试数据目录；输出是配对样本表和未配对文件表。若没有配对结果，请检查文件命名是否满足 `multi_<运动拼音数字>.csv` 和 `multi_<运动拼音数字>_ref.csv`。

In [ ]:
# 这一步扫描文件并配对。
# 输入：TESTDATA_DIR 下的 CSV 文件。
# 输出：pairs_df 和 unpaired_df。
# 若失败：检查文件名和目录权限。
discovery = discover_sample_pairs_with_unpaired(TESTDATA_DIR)
pairs_df = pd.DataFrame([
    {"sample": p.stem, "sensor_csv": str(p.sensor_csv), "ref_csv": str(p.ref_csv)}
    for p in discovery.pairs
])
unpaired_df = pd.DataFrame([
    {"file_name": u.file_name, "file_path": str(u.file_path), "reason": u.reason}
    for u in discovery.unpaired
])
display(pairs_df)
display(unpaired_df)
# 本 cell 结束后，只有 pairs_df 中的样本会进入 QC。

# 5. 原始数据质量过滤 QC

本节按 Ut1/Ut2 前 10 秒的去基线 STD 与离群点比例规则筛选好采样。输入是配对样本；输出是包含 `group_id`、文件路径、判坏原因和 QC 指标的表格。若样本被判坏，它会在后续批处理中跳过，并写入 `bad_samples.csv` 与 `batch_summary.csv`。

In [ ]:
# 这一步执行原始数据质量过滤。
# 输入：discovery.pairs。
# 输出：qc_df 和 good_pairs。
# 若失败：检查 Ut1(mV)/Ut2(mV) 列名和 CSV 可读性。
qc_rows = []
good_pairs = []
for pair in discovery.pairs:
    qc = quality_filter_sample(pair.sensor_csv, fs=100, group_id=pair.motion_id, ref_csv=pair.ref_csv)
    qc_rows.append(qc.to_dict())
    if qc.is_good:
        good_pairs.append(pair)

qc_df = pd.DataFrame(qc_rows)
display(qc_df)
print(f"好采样 {len(good_pairs)} / 配对样本 {len(discovery.pairs)}")
# 本 cell 结束后，good_pairs 中的样本会进入预处理、分段、对齐和优化。

# 6. 预处理、运动分段与时间对齐预览

本节用第一组好采样做快速预览，确认时间轴重建、缺失值处理、PPG 毛刺修复、CF 计算、带通滤波、重采样、运动/静息/恢复分段和 PPG-HR 对齐可以执行。输入是 QC 通过的样本；输出是分段与对齐摘要。若失败，请检查 ACC 运动段是否足够明显，或参考 HR 是否可解析。

In [ ]:
# 这一步预跑单个好采样的预处理、分段和对齐。
# 输入：good_pairs[0]、Fs_Target=100、TW=8。
# 输出：preview_summary。
# 若失败：查看 segmentation reason 或 alignment 异常信息。
preview_summary = None
if good_pairs:
    pair = good_pairs[0]
    ds = load_and_preprocess_protocol(pair.sensor_csv, pair.ref_csv, fs_origin=100)
    ds = resample_protocol_dataset(ds, fs_target=100)
    seg = detect_activity_segments(ds.accx, ds.accy, ds.accz, ds.fs, TW=8)
    if not seg.is_valid:
        raise RuntimeError(f"分段失败: {pair.stem}: {seg.reason}")
    aligned = align_ppg_to_ref_hr(ds, seg, TW=8, fs_target=ds.fs)
    preview_summary = {
        "sample": pair.stem,
        "fs": ds.fs,
        "motion_start_s": aligned.segment_info.motion_start_s,
        "motion_end_s": aligned.segment_info.motion_end_s,
        "best_tdelay_s": aligned.alignment_info.best_tdelay_s,
        "num_windows": aligned.alignment_info.num_windows,
        "rest_windows": len(aligned.rest_indices),
        "motion_windows": len(aligned.motion_indices),
        "recovery_windows": len(aligned.recovery_indices),
    }
    display(pd.DataFrame([preview_summary]))
else:
    print("没有 QC 通过的样本，后续批处理会只写跳过记录。")
# 本 cell 结束后，至少一组好采样应能完成分段和对齐。

# 7. Debug / Formal 贝叶斯优化配置

本节设置 debug/smoke 与 formal 预算。输入是人工选择的 `DEBUG_MODE`；输出是 `MAX_ITERATIONS`、`NUM_REPEATS`、`N_JOBS` 和进度打印间隔。默认使用 debug 小预算完整跑 14 个模式，每个模式仍保留 3 个 repeat，用来验证 repeat 逻辑。

In [ ]:
# 这一步配置优化预算和进度打印频率。
# 输入：DEBUG_MODE。
# 输出：MAX_ITERATIONS、NUM_REPEATS、N_JOBS、PROGRESS_EVERY_N_TRIALS、search_space。
# 若失败：检查参数是否为正整数，Windows/Jupyter debug 模式建议 N_JOBS=1。
DEBUG_MODE = False
QUICK_TEST_N_TRIALS = 1
FORMAL_N_TRIALS = 200

if DEBUG_MODE:
    # 调试模式：每个目标/方案做 3 个 repeat，每个 repeat 只跑少量 trial，便于先验证全流程。
    MAX_ITERATIONS = QUICK_TEST_N_TRIALS
    NUM_REPEATS = 1
    N_JOBS = 1
    PROGRESS_EVERY_N_TRIALS = 1
else:
    # 正式模式：每个目标/方案做 3 个 repeat，每个 repeat 默认 250 trials。
    MAX_ITERATIONS = FORMAL_N_TRIALS
    NUM_REPEATS = 2
    N_JOBS = None
    PROGRESS_EVERY_N_TRIALS = 20

search_space = ProtocolSearchParams()
print({
    "DEBUG_MODE": DEBUG_MODE,
    "MAX_ITERATIONS": MAX_ITERATIONS,
    "NUM_REPEATS": NUM_REPEATS,
    "N_JOBS": N_JOBS,
    "PROGRESS_EVERY_N_TRIALS": PROGRESS_EVERY_N_TRIALS,
    "LMS_Mu_Base": 0.01,
    "LMS_Mu_Min": 1e-5,
})
# 本 cell 结束后，批处理会用上述预算运行完整 2 目标 x 7 方案。

# 8. 批量运行 14 组训练模式

本节调用主入口，完成文件配对、QC、预处理、分段、对齐、14 模式小预算 Optuna 或随机 fallback 优化、CSV/JSON/PNG 输出。进度会显示当前训练到第几组文件、第几个模式、第几个 repeat 和第几个 trial。若失败，请看进度打印中的 stage、sample、mode、trial 和 reason。

In [ ]:
# 这一步运行完整批处理，并在 Notebook 中打印训练进度。
# 输入：TESTDATA_DIR、CSV_OUT_DIR、REPORT_OUT_DIR、FIG_OUT_DIR、优化预算。
# 输出：result 对象、batch_summary.csv、每个成功样本的 CSV/JSON/PNG。
# 若失败：优先检查最后一次打印的 sample、mode_idx、target_scope、cascade_scheme、trial_idx。
def notebook_progress(info):
    stage = info.get("stage", "")
    if stage == "optimization":
        trial_idx = int(info.get("trial_idx") or 0)
        trial_total = int(info.get("trial_total") or 0)
        should_print = (
            trial_idx == 1
            or trial_idx == trial_total
            or (PROGRESS_EVERY_N_TRIALS > 0 and trial_idx % PROGRESS_EVERY_N_TRIALS == 0)
        )
        if should_print:
            print(
                f"文件 {info.get('sample_idx')}/{info.get('sample_total')} | "
                f"{info.get('sample')} | "
                f"模式 {info.get('mode_idx')}/{info.get('mode_total')} | "
                f"{info.get('target_scope_value')} / {info.get('cascade_scheme')} | "
                f"repeat {info.get('repeat_idx')}/{info.get('repeat_total')} | "
                f"trial {trial_idx}/{trial_total} | "
                f"本次 AAE {info.get('value')} | "
                f"历史最优 AAE {info.get('best_aae')}"
            )
    elif stage == "optimization_mode":
        print(
            f"进入模式 {info.get('mode_idx')}/{info.get('mode_total')}: "
            f"{info.get('target_scope_value')} / {info.get('cascade_scheme')}"
        )
    elif stage == "qc":
        print(f"QC {info.get('current')}/{info.get('total')}: {info.get('sample')}")
    elif stage == "sample":
        print(f"开始样本 {info.get('sample_idx')}/{info.get('sample_total')}: {info.get('sample')}")
    elif stage == "output":
        print(f"完成样本 {info.get('sample_idx')}/{info.get('sample_total')}: {info.get('sample')}")
        print("CSV:", info.get("result_csv"))
        print("JSON:", info.get("report_json"))

result = run_batch_adaptive_protocol(
    input_dir=TESTDATA_DIR,
    csv_out_dir=CSV_OUT_DIR,
    report_out_dir=REPORT_OUT_DIR,
    fig_out_dir=FIG_OUT_DIR,
    max_iterations=MAX_ITERATIONS,
    num_repeats=NUM_REPEATS,
    random_state=142,
    num_seed_points=10,
    fs_origin=100,
    n_jobs=N_JOBS,
    debug_mode=DEBUG_MODE,
    search_space=search_space,
    verbose=True,
    on_log=print,
    progress_callback=notebook_progress,
)

print("batch summary:", result.batch_summary_csv)
# 本 cell 结束后，输出目录中应出现 batch_summary、QC 表、成功样本 CSV/JSON/PNG。

# 9. 结果汇总与跳过原因检查

本节读取批处理输出的 summary 与 QC 表，检查成功、跳过或失败状态。输入是 CSV 输出目录；输出是显示表格。若某样本被跳过，`batch_summary.csv` 的 reason 会说明 QC、配对、分段、对齐或依赖缺失等原因。

In [ ]:
# 这一步汇总输出表。
# 输入：CSV_OUT_DIR 下的 qc_summary.csv、good_samples.csv、bad_samples.csv、batch_summary.csv。
# 输出：Notebook 中显示的结果表。
# 若失败：检查阶段 8 是否完成，或输出目录是否被清理。
summary_paths = [
    CSV_OUT_DIR / "qc_summary.csv",
    CSV_OUT_DIR / "good_samples.csv",
    CSV_OUT_DIR / "batch_summary.csv",
]
for path in summary_paths:
    print("===", path.name, "===")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("缺失:", path)
# 本 cell 结束后，可以确认两组 testdata 的 ok/skipped 状态。

# 10. 图表预览与结果检查

本节检查每个成功样本是否生成 1 张 13 路信号图、2 张贝叶斯优化训练曲线图、2 张 HR 对比图，以及 CSV/JSON 是否覆盖 14 个模式。输入是 `result.sample_outputs`；输出是在 Notebook 中展示的文件路径、表格预览和图片预览。若数量不足，请回到第 8 节查看对应样本或模式的错误信息。

In [ ]:
# 这一步检查每个成功样本的 CSV/JSON/PNG 是否齐全，并在 Notebook 中预览图片。
# 输入：result.sample_outputs。
# 输出：每个样本的输出路径、CSV/JSON 模式数量和 PNG 预览。
# 若失败：检查 FILTERED_SIGNAL_OUT_DIR、HR_COMPARE_OUT_DIR、BAYES_CURVE_OUT_DIR、REPORT_OUT_DIR 和 CSV_OUT_DIR。
for sample, paths in result.sample_outputs.items():
    print("样本:", sample)
    print("CSV:", paths.result_csv)
    print("JSON:", paths.report_json)
    print("运动段 13 路信号 PNG:", paths.signal_png)
    print("运动段贝叶斯训练曲线 PNG:", paths.bayes_motion_png)
    print("运动+恢复段贝叶斯训练曲线 PNG:", paths.bayes_motion_recovery_png)
    print("运动段 HR 对比 PNG:", paths.motion_png)
    print("运动+恢复段 HR 对比 PNG:", paths.motion_recovery_png)

    df = pd.read_csv(paths.result_csv)
    with open(paths.report_json, "r", encoding="utf-8") as f:
        payload = json.load(f)
    mode_count_csv = df[["target_scope", "cascade_scheme"]].drop_duplicates().shape[0]
    print("CSV 模式数:", mode_count_csv)
    print("JSON 模式数:", len(payload))
    display(df.head())
    for image_path in [
        paths.signal_png,
        paths.bayes_motion_png,
        paths.bayes_motion_recovery_png,
        paths.motion_png,
        paths.motion_recovery_png,
    ]:
        if image_path is not None and Path(image_path).exists():
            display(Image(filename=str(image_path)))

if not result.sample_outputs:
    print("没有成功输出的样本，请查看 batch_summary.csv 中的跳过原因。")
# 本 cell 结束后，debug 流程的输出完整性应能被人工快速核验。